# Nested Learning for CAD — HOPE-CAD v5.1 normal pilot

Method mới `hope_cad_v5_1_causal_scaled_utility`, **chưa có T4 pilot hoặc AD gain**. Đọc `docs/main_spec/v5_1_implementation.md` và các full specs/acceptance v5.1 trong source.

Ba inputs: `hope_cad_support_v5_1_source_r1.zip` (hoặc Dataset Kaggle đã giải nén đúng ZIP này), MVTec AD, ViT-B/8 `vit_base_patch8_224.augreg_in21k.safetensors`. **Không attach learned evidence v5/W6/W7/recovery.** Tháo source cũ khác version.

Settings GPU T4 x2; dùng **Save Version → Save & Run All**. `PHASE='pilot'` tự chạy preflight24 real train normals trên discarded clone, rồi bốn fresh jobs CADIC/GEOMETRIC/FIRST_ORDER/NESTED, mỗi job256 normals. Chỉ extract train/good; không test/GT. Không search hoặc tăng training budget để cứu gate.

`SESSION_HOURS=3.0` là soft execution limit, không phải ETA. Có thể đặt9.0 nếu đủ quota; thay soft limit không đổi256 images/method. Có heartbeat30giây. Pause thì tải ZIP/log, attach **latest v5.1 output** cùng đúng source/MVTec/ViT và chạy tiếp; tháo output phiên cũ hơn. Resume phải giữ source/runtime/protocol/method identity. Muốn preflight riêng: `PHASE='preflight'`; PASS rồi attach output đó và chuyển pilot.

`pilot_complete_pending_review` là complete chờ nghiệm thu, chưa auto benchmark. `pilot_complete_scientific_gate_failed` là complete nhưng scientific gate FAIL: giữ ZIP/log và stop review. `paused_at_image_boundary` chưa complete, cần resume. Mọi technical failure đều xuất failure ZIP nếu process còn chạy; hard platform kill có thể ngăn export. Không pip-upgrade runtime để qua guard.

Output `/kaggle/working/hope_cad_support_v5_1_pilot_evidence.zip`. Giữ Output của Saved Version và tải ZIP/log về `.agents/artifacts/runs/support_v5_1_pilot_01/session_01/` (các lần resume tăng session). Local tests không thay canonical T4 acceptance.


In [ ]:
PHASE='pilot'  # pilot: preflight tích hợp + 4 jobs x256 normals; preflight: chỉ kiểm trước
SESSION_HOURS=3.0  # soft stop; lưu/resume ở atomic boundary; không phải ETA
SOURCE_OVERRIDE=None
RESUME_OVERRIDE=None   # chỉ latest support-v5.1 pilot output; lần đầu để None
MVTEC_OVERRIDE=None
CHECKPOINT_OVERRIDE=None

from pathlib import Path,PurePosixPath
import hashlib,json,zipfile,os,sys,time,shutil
STARTED=time.monotonic()
INPUT=Path('/kaggle/input');ROOT=Path('/tmp/hope_support_v5_1_source')
print('[BOOTSTRAP] Tìm source HOPE-CAD v5',flush=True)
def hash_file(p):
    with Path(p).open('rb') as stream:return hashlib.file_digest(stream,'sha256').hexdigest()
def source_manifest(p):
    if p.is_dir():return json.loads((p/'bundle_manifest.json').read_text())
    with zipfile.ZipFile(p) as z:
        if len(z.namelist())!=len(set(z.namelist())):raise ValueError('Duplicate source ZIP paths')
        m=json.loads(z.read('bundle_manifest.json'))
        if set(z.namelist())!=set(m['files'])|{'bundle_manifest.json'}:raise ValueError('Source inventory mismatch')
        return m
def target(name):
    rel=PurePosixPath(name)
    if rel.is_absolute() or '..' in rel.parts or rel.as_posix()!=name or '\\' in name or ':' in name:raise ValueError('Unsafe source path')
    p=ROOT/name
    if not p.resolve().is_relative_to(ROOT.resolve()):raise ValueError('Escaping source path')
    return p
if SOURCE_OVERRIDE:choices=[Path(SOURCE_OVERRIDE)]
else:
    choices=[]
    for p in INPUT.rglob('bundle_manifest.json'):
        if json.loads(p.read_text()).get('schema')=='hope_support_v5_1_source_v1':choices.append(p.parent)
    if not choices:
        for p in INPUT.rglob('*.zip'):
            try:
                if source_manifest(p).get('schema')=='hope_support_v5_1_source_v1':choices.append(p)
            except (KeyError,ValueError,zipfile.BadZipFile):pass
if len(choices)!=1:raise RuntimeError(f'Expected ONE support-v5.1 source, found {choices}; set SOURCE_OVERRIDE')
source=choices[0];manifest=source_manifest(source)
if manifest['schema']!='hope_support_v5_1_source_v1':raise ValueError('Wrong source schema')
if ROOT.exists() and any(ROOT.iterdir()):
    previous=ROOT/'bundle_manifest.json'
    if not previous.is_file() or json.loads(previous.read_text()).get('files')!=manifest['files']:
        raise RuntimeError('Stale source directory: start a fresh kernel/saved version for a different delivery')
    existing={p.relative_to(ROOT).as_posix() for p in ROOT.rglob('*') if p.is_file() and '__pycache__' not in p.parts}
    if existing!=set(manifest['files'])|{'bundle_manifest.json'}:
        raise RuntimeError('Unexpected files in source directory; use a fresh kernel/saved version')
ROOT.mkdir(parents=True,exist_ok=True)
if source.is_dir():
    for name,expected in manifest['files'].items():
        original=source/name;dest=target(name)
        if not original.is_file() or original.is_symlink() or hash_file(original)!=expected:raise ValueError('Source checksum mismatch: '+name)
        dest.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(original,dest)
else:
    with zipfile.ZipFile(source) as z:
        for name,expected in manifest['files'].items():
            dest=target(name);dest.parent.mkdir(parents=True,exist_ok=True)
            with z.open(name) as reader,dest.open('wb') as writer:shutil.copyfileobj(reader,writer)
            if hash_file(dest)!=expected:raise ValueError('Source checksum mismatch: '+name)
(ROOT/'bundle_manifest.json').write_text(json.dumps(manifest,sort_keys=True))
os.chdir(ROOT);sys.path.insert(0,str(ROOT));sys.dont_write_bytecode=True
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
print('[BOOTSTRAP] Verified source files:',len(manifest['files']),manifest['origin'],flush=True)
WORK=Path('/kaggle/working/hope_support_v5_1_pilot')
ASSETS=Path('/tmp/hope_support_v5_1_assets')
OUTPUT=Path('/kaggle/working/hope_cad_support_v5_1_pilot_evidence.zip')


In [ ]:
from notebook_support.support_v5_1.entry import run_notebook
RESULT=run_notebook(dict(phase=PHASE,session_hours=SESSION_HOURS,started=STARTED,
    inputs=str(INPUT),work=str(WORK),assets=str(ASSETS),output=str(OUTPUT),
    RESUME_OVERRIDE=RESUME_OVERRIDE,MVTEC_OVERRIDE=MVTEC_OVERRIDE,CHECKPOINT_OVERRIDE=CHECKPOINT_OVERRIDE))
